In [28]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score


In [1]:

df = pd.read_csv('/content/bank-full.csv', sep=';')

df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [3]:
print(df['education'].unique())

['tertiary' 'secondary' 'unknown' 'primary']


In [4]:
education_map = {
    'unknown':0,
    'primary':1,
    'secondary':2,
    'tertiary':3
}
df['education'] = df['education'].map(education_map)
print(df['education'].head())

0    3
1    2
2    2
3    0
4    0
Name: education, dtype: int64


In [5]:
df['age_balance'] = df['age'] * df['balance']

df['duration_balance'] = (
    df['duration'] * df['balance']
)

df['age_education'] = (
    df['age'] * df['education']
)

df['duration_education'] = (
    df['duration'] * df['education']
)

In [6]:
df.isnull().sum()

,0
age,0
job,0
marital,0
education,0
default,0
balance,0
housing,0
loan,0
contact,0
day,0


In [7]:
df['y'].value_counts()

,count
y,
no,39922
yes,5289


In [8]:
#seperate the features and targe
X = df.drop('y', axis=1)
y= df['y']

In [11]:
#Encode the target variable
le = LabelEncoder()
y = le.fit_transform(y)
print("Encoded Target Values:")
print(np.unique(y))

Encoded Target Values:
[0 1]


In [12]:
df.columns

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y', 'age_balance', 'duration_balance',
       'age_education', 'duration_education'],
      dtype='object')

In [13]:
df.shape

(45211, 21)

In [14]:
X = pd.get_dummies(
    X,
    columns=[
        'job',
        'marital',
        'default',
        'housing',
        'loan',
        'contact',
        'month',
        'poutcome'
    ],
    drop_first=True)

In [15]:
print(X.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 44 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   age                 45211 non-null  int64
 1   education           45211 non-null  int64
 2   balance             45211 non-null  int64
 3   day                 45211 non-null  int64
 4   duration            45211 non-null  int64
 5   campaign            45211 non-null  int64
 6   pdays               45211 non-null  int64
 7   previous            45211 non-null  int64
 8   age_balance         45211 non-null  int64
 9   duration_balance    45211 non-null  int64
 10  age_education       45211 non-null  int64
 11  duration_education  45211 non-null  int64
 12  job_blue-collar     45211 non-null  bool 
 13  job_entrepreneur    45211 non-null  bool 
 14  job_housemaid       45211 non-null  bool 
 15  job_management      45211 non-null  bool 
 16  job_retired         45211 non-null  bool

In [16]:
X.columns

Index(['age', 'education', 'balance', 'day', 'duration', 'campaign', 'pdays',
       'previous', 'age_balance', 'duration_balance', 'age_education',
       'duration_education', 'job_blue-collar', 'job_entrepreneur',
       'job_housemaid', 'job_management', 'job_retired', 'job_self-employed',
       'job_services', 'job_student', 'job_technician', 'job_unemployed',
       'job_unknown', 'marital_married', 'marital_single', 'default_yes',
       'housing_yes', 'loan_yes', 'contact_telephone', 'contact_unknown',
       'month_aug', 'month_dec', 'month_feb', 'month_jan', 'month_jul',
       'month_jun', 'month_mar', 'month_may', 'month_nov', 'month_oct',
       'month_sep', 'poutcome_other', 'poutcome_success', 'poutcome_unknown'],
      dtype='object')

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (36168, 44)
Testing shape: (9043, 44)


In [18]:
X_train_lr = X_train.copy()
X_test_lr = X_test.copy()

In [21]:
num_cols = [
    'age',
    'education',
    'balance',
    'day',
    'duration',
    'campaign',
    'pdays',
    'previous',
    'age_balance',
    'duration_balance',
    'age_education',
    'duration_education'
]

scaler = StandardScaler()
X_train_lr[num_cols] = scaler.fit_transform(X_train_lr[num_cols])
X_test_lr[num_cols] =  scaler.transform(X_test_lr[num_cols])

In [22]:
scaler = StandardScaler()

X_train_lr[num_cols] = scaler.fit_transform(X_train_lr[num_cols])
X_test_lr[num_cols] = scaler.transform(X_test_lr[num_cols])

In [23]:
print(X_train_lr.shape)
print(X_test_lr.shape)

(36168, 44)
(9043, 44)


In [26]:
ann_model = Sequential([
    Dense(64, activation='relu', input_shape=(44,)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

ann_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

ann_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         2,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,505 (21.50 KB)

 Trainable params: 5,505 (21.50 KB)

 Non-trainable params: 0 (0.00 B)

In [27]:
history = ann_model.fit(
    X_train_lr.astype('float32'),
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20
905/905 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8955 - loss: 0.2448 - val_accuracy: 0.9024 - val_loss: 0.2176
Epoch 2/20
905/905 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9027 - loss: 0.2144 - val_accuracy: 0.9039 - val_loss: 0.2089
Epoch 3/20
905/905 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9055 - loss: 0.2076 - val_accuracy: 0.9057 - val_loss: 0.2089
Epoch 4/20
905/905 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9083 - loss: 0.2017 - val_accuracy: 0.9066 - val_loss: 0.2062
Epoch 5/20
905/905 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9095 - loss: 0.1973 - val_accuracy: 0.9064 - val_loss: 0.2042
Epoch 6/20
905/905 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9117 - loss: 0.1935 - val_accuracy: 0.9041 - val_loss: 0.2072
Epoch 7/20
905/905 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9119 - loss: 0.1906 - val_accuracy: 0.9079 - val_loss: 0.2038
Epoch 8/20
905/905 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9145 - loss: 0.1868 - val_accuracy: 0.

In [29]:
y_prob_ann = ann_model.predict(
    X_test_lr.astype('float32')
)

y_pred_ann = (y_prob_ann > 0.5).astype(int)

283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [30]:
print("Accuracy:", accuracy_score(y_test, y_pred_ann))
print(classification_report(y_test, y_pred_ann))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_ann))

Accuracy: 0.9047882339931439
              precision    recall  f1-score   support

           0       0.95      0.95      0.95      7985
           1       0.59      0.59      0.59      1058

    accuracy                           0.90      9043
   macro avg       0.77      0.77      0.77      9043
weighted avg       0.90      0.90      0.90      9043

ROC-AUC: 0.9215939503771842
